# Creating a PubMed Chatbot on GCP

## Overview

For this tutorial we create a PubMed chatbot that will answer questions by gathering information from documents we have provided via an index. The model we will be using today is the pretrained 'gemini-2.5-flash' model from GCP.

## Learning Objectives
- Set up a Retrieval-Augmented Generation (RAG) architecture on GCP.
- Collect, chunk, and embed biomedical documents into Vertex AI Vector Search.
- Build a Python terminal application running a conversational LangChain interface powered by Gemini 2.5 Flash.

## Prerequisites
You need access to Verted AI. If you want to deploy a model (see below) then follow those instructions.

## Get Started

### Optional: Deploy the Model

In this tutorial we will be using a Google's gemini model **gemini-2.0-flash** which doesn't need to be deployed but if you would like to use another model you choose one from the **Model Garden** using the console which will allow you to add a model to your model registry, create an endpoint (or use an existing one), and deploy the model all in one step.

### PubMed API vs RAG with Vertex AI Vector Search

Our chatbot will rely on documents to answer our questions to do so we are supplying it a **vector index**. A vector index or index is a data structure that enables fast and accurate search and retrieval of vector embeddings from a large dataset of objects. We will be working with two options for our index: PubMed API vs RAG Vertex AI Vector Search method.

**What is the difference?**

The **PubMed API** is provided free by langchain to connect your model to more than **35 million citations** for biomedical literature from MEDLINE, life science journals, and online books. The langchain package for PubMed is already a retriever meaning that just simply using this tool will our chatbot beable to retrieve documents to refer to. 

**Vertex AI Vector Search** (formally known as Matching Engine) is a vector store from GCP that allows the user more **security and control** on which documents you wish to supply to your model. Vector Search, formerly known as Vertex AI Matching Engine, is a vector store or database that stores the **embeddings** of your documents and the metadata. Because this is not a retriever we have to make it so for our model to send back an output that also tells us which documents it is referencing, this is where RAG comes in. **RAG** stands for **Retrieval-augmented generation** it is a method or technique that **indexes documents** by first loading them in, splitting them into chucks (making it easier for our model to search for relevant splits), embedding the splits, then storing them in a vector store. The next steps in RAG are based on the question you ask your chatbot. If we were to ask it "What is a cell?" the vector store will be searched by a retriever to find relevant splits that have to do with our question, thus **retrieving relevant documents**. And finally our chatbot will **generate an answer** that makes sense of what a cell is, as part of the answer it will also point out which source documents it used to create the answer.

We will be exploring both methods!

### Step 1: Install Required Libraries

Open your system terminal and execute this line to download the fast, local in-memory vector storage framework (faiss-cpu) along with your LangChain modules:

In [ ]:
!pip install langchain-core langchain-google-vertexai langchain-community faiss-cpu biopython

### Step 2: Set Up Authentication

Run this inside your terminal session to point the script to your active Google Cloud project ID.

In [ ]:
PROJECT_ID = subprocess.check_output("gcloud config get-value project", shell=True).decode("utf-8").strip()

Make sure you have previously logged in using your cloud CLI locally (gcloud auth application-default login), so LangChain can reach the Gemini 2.5 Flash model safely.

### Step 3: Fetch PubMed Data Instantly

Instead of managing complex database buckets, fetch medical abstracts directly into your application space.

In [ ]:
import os
from Bio import Entrez

print(" Fetching recent research abstracts from PubMed...")
Entrez.email = "your-email-address"  # Change to your email address string

# Search PubMed database for matching article keys
search_handle = Entrez.esearch(db="pubmed", term="brain cancer", retmax=10)
search_results = Entrez.read(search_handle)
search_handle.close()

# Fetch target textual context content lines
fetch_handle = Entrez.efetch(db="pubmed", id=search_results["IdList"], rettype="abstract", retmode="text")
abstracts_data = fetch_handle.read()
fetch_handle.close()

# Parse down abstracts data chunks into a clean text array list 
raw_docs = [abs.strip() for abs in abstracts_data.split("\n\n\n") if abs.strip()]


Limiting the initial query to 10 articles speeds up execution so you can verify that the retrieval logic works without system lag.

### Step 4: Vectorize Documents in RAM

It uses the text-embedding-005 engine to convert the abstracts into mathematical vectors right inside your system RAM—skipping complex cloud database architectures.

In [ ]:
from langchain_google_vertexai import VertexAIEmbeddings
from langchain_community.vectorstores import FAISS

print(" Initializing fast In-Memory Vector Store via FAISS...")
embeddings = VertexAIEmbeddings(model_name="text-embedding-005")

# Generate mathematical vector array locations directly in system memory
vector_store = FAISS.from_texts(texts=raw_docs, embedding=embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})


The variable retriever pulls the top 3 most relevant context document snippets matching your prompt every time you ask a question.

### Step 5: Construct the RAG Pipeline

Tie your data retriever to the gemini-2.5-flash model using LangChain's modern chain builders. Add this chunk to build the retrieval chain components. This section includes the Zero-Shot Prompt strategy to keep processing times immediate and accurate.

In [ ]:
!pip install langchain-classic

In [ ]:
import os
from Bio import Entrez

# Modern LangChain Core & Google Vertex imports
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_google_vertexai import ChatVertexAI, VertexAIEmbeddings
from langchain_community.vectorstores import FAISS

# Updated Classic paths (Fixes the 'No module named langchain.chains' error)
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever


llm = ChatVertexAI(model_name="gemini-2.5-flash", temperature=0.1)

context_prompt = ChatPromptTemplate.from_messages([
    ("system", "Formulate a standalone question from the chat history and latest user question."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
history_aware_retriever = create_history_aware_retriever(llm, retriever, context_prompt)


# --- PROMPT DEFINITIONS ---

zero_shot_prompt = (
    "Instruction: You are an expert medical research assistant.\n"
    "Create a highly concise, objective summary from the provided research context documents.\n"
    "Do not include personal interpretations, opinions, or unverified assertions. "
    "Write entirely in your own words. "
    "First provide the summary, then list source citations at the very end.\n\n"
    "Answer 'I don't know' if the answer cannot be found in the context documents.\n\n"
    "Context:\n{context}"
)

multi_shot_prompt = (
    "Instruction: You are an expert medical research assistant. Summarize the context objectively.\n"
    "First provide a concise summary, then list source citations at the very end. "
    "Answer 'I don't know' if the answer cannot be found in the context documents.\n\n"
    "Example 1 (Successful Context Synthesis):\n"
    "Question: What is the primary therapy for brain tumors?\n"
    "Context: Surgery remains the cornerstone of management for brain tumors. Post-operative care frequently includes radiation therapy.\n"
    "Answer:\n"
    "The baseline therapy for brain tumors is surgical intervention, which is frequently paired with post-operative radiation care to manage recurrence risk.\n"
    "Sources: Context Document 1\n\n"
    "Example 2 (Missing Information Handler):\n"
    "Question: Does Vitamin C cure brain cancer?\n"
    "Context: Studies show that standard chemotherapies extend survival rates. Diet adjustments are monitored closely.\n"
    "Answer:\n"
    "I don't know.\n"
    "Sources: None\n\n"
    "Actual Context To Analyze:\n{context}"
)


# --- TOGGLE SWITCH CONTROLLER ---
# Change this string to "ZERO" or "MULTI" depending on your needs [2026 Strategy]
PROMPT_MODE = "ZERO" 

if PROMPT_MODE == "MULTI":
    system_prompt = multi_shot_prompt
    print(" Pipeline utilizing Multi-Shot formatting style...")
else:
    system_prompt = zero_shot_prompt
    print(" Pipeline utilizing Zero-Shot formatting style...")


# --- BUILD MAIN PIPELINE OBJECTS ---
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

document_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, document_chain)


### Step 6: Flat Single-Shot Q&A Execution

Paste this final piece at the very bottom of your script file. This explicitly triggers a question against your loaded text data data, catches the answer, and prints it out to your screen instantly.

In [ ]:
# 1. Define your specific medical research query directly
my_question = "What are the primary treatment methods or challenges discussed regarding brain cancer?"

print(f"\n Sending Question: '{my_question}'")
print(" Processing through RAG pipeline...")

# 2. Fire the query through the pipeline using an empty history list
response = rag_chain.invoke({
    "input": my_question, 
    "chat_history": []  
})

# 3. Output the result cleanly to the terminal screen
print("\n Answer:")
print(response["answer"])


## Conclusion
Here you learned how to deploy and interact with a chat model and also how to deploy an inference script to create an interactive chatbot in the terminal.